# 03 - Modelado y evaluacion de modelos

Este notebook corresponde a la **Persona 4 - Modelado**. La meta es comparar varios modelos de regresion para predecir `num_bikes_available`, es decir, el numero de bicicletas disponibles en una estacion BiciMAD.

El notebook compara:

1. Baseline con la media.
2. Regresion lineal.
3. Ridge.
4. Random Forest Regressor.

Tras la correccion del dataset, `snapshot_hour` contiene las 24 horas del dia, asi que el modelo ya puede aprender patrones horarios reales.

## Resumen de resultados actualizados

Con el dataset regenerado y la variable horaria corregida, el notebook utiliza una muestra reproducible de **120.000 filas** y un conjunto de test de **24.000 filas**.

En la ultima ejecucion, el mejor modelo fue **Random Forest**, con estos resultados aproximados:

| Modelo | MAE | RMSE | R2 |
|---|---:|---:|---:|
| Random Forest | 3.63 | 4.55 | 0.427 |
| Ridge | 4.39 | 5.37 | 0.201 |
| Regresion lineal | 4.39 | 5.37 | 0.200 |
| Baseline media | 4.99 | 6.01 | -0.000 |

La correccion de las horas mejora claramente el modelado: Random Forest aprovecha la informacion horaria y reduce el error frente a la version anterior.

## 1. Importacion de librerias y configuracion

Cargamos las librerias necesarias para trabajar con datos, construir pipelines y evaluar modelos. Tambien dejamos rutas y parametros en variables visibles para que el notebook sea facil de modificar.

In [ ]:
# Librerias de sistema y control de avisos
from pathlib import Path
import warnings

# Librerias de datos
import joblib
import numpy as np
import pandas as pd

# Herramientas de scikit-learn
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

warnings.filterwarnings("ignore")

# Rutas del proyecto.
# Esta deteccion permite ejecutar el notebook desde la raiz del repo o desde la carpeta notebooks.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "station_status_history_2022_modeling_base_clean.csv"
MODEL_OUTPUT_PATH = PROJECT_ROOT / "models" / "modelo_final_definitivo_random_forest.joblib"

# Parametros reproducibles
RANDOM_STATE = 42
SAMPLE_SIZE = 120_000
TEST_SIZE = 0.20

## 2. Carga de datos

Usamos el dataset limpio generado por las fases anteriores. Para ahorrar memoria, cargamos solo las columnas necesarias para modelado.

In [ ]:
# Variable objetivo: lo que queremos predecir
TARGET = "num_bikes_available"

# Variables numericas usadas como entrada del modelo
numeric_features = [
    "snapshot_hour",
    "snapshot_day_of_week",
    "snapshot_month",
    "station_capacity",
    "station_latitude",
    "station_longitude",
    "weather_temperature_mean_c",
    "weather_temperature_min_c",
    "weather_temperature_max_c",
    "weather_precipitation_mm",
    "weather_humidity_mean_pct",
]

# Variables categoricas o booleanas usadas como entrada del modelo
categorical_features = [
    "station_id_historical",
    "day_type",
    "is_working_day",
    "is_weekend",
    "weather_has_precipitation",
]

# Columnas totales a cargar
usecols = [TARGET] + numeric_features + categorical_features

# Carga del CSV limpio
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"No se encuentra el dataset limpio en {DATA_PATH}. "
        "Hay que descargarlo con Git LFS o regenerarlo desde los notebooks anteriores antes de ejecutar el modelado."
    )

df = pd.read_csv(DATA_PATH, usecols=usecols)

print(f"Filas totales del dataset limpio: {len(df):,}")
print(f"Columnas usadas en modelado: {len(usecols)}")
print(f"Media del target ({TARGET}): {df[TARGET].mean():.2f} bicicletas")
print(f"Rango del target: {df[TARGET].min():.0f} - {df[TARGET].max():.0f} bicicletas")
print(f"Valores unicos de snapshot_hour: {sorted(df['snapshot_hour'].dropna().unique().tolist())}")
print(f"Meses disponibles: {sorted(df['snapshot_month'].dropna().unique().tolist())}")

df.head()

Filas totales del dataset limpio: 2,306,832
Columnas usadas en modelado: 17
Media del target (num_bikes_available): 9.77 bicicletas
Rango del target: 0 - 30 bicicletas
Valores unicos de snapshot_hour: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
Meses disponibles: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]


,station_id_historical,station_latitude,station_longitude,station_capacity,num_bikes_available,snapshot_hour,snapshot_day_of_week,snapshot_month,day_type,is_working_day,is_weekend,weather_temperature_mean_c,weather_temperature_min_c,weather_temperature_max_c,weather_precipitation_mm,weather_humidity_mean_pct,weather_has_precipitation
0,1,40.417214,-3.701834,30,0,0,5,1,festivo,False,True,10.475,3.025,17.9,0.0,61.0,False
1,2,40.417313,-3.701603,30,0,0,5,1,festivo,False,True,10.475,3.025,17.9,0.0,61.0,False
2,3,40.420589,-3.705842,24,7,0,5,1,festivo,False,True,10.475,3.025,17.9,0.0,61.0,False
3,4,40.430294,-3.706917,18,14,0,5,1,festivo,False,True,10.475,3.025,17.9,0.0,61.0,False
4,5,40.428552,-3.702587,24,17,0,5,1,festivo,False,True,10.475,3.025,17.9,0.0,61.0,False


## 3. Validacion previa

Antes de entrenar, revisamos si hay nulos y comprobamos la calidad de la variable hora. Esta comprobacion confirma que el dataset regenerado contiene las 24 horas del dia.

In [ ]:
# Porcentaje de nulos por columna
null_report = (
    df.isna()
    .mean()
    .mul(100)
    .round(2)
    .rename("pct_nulos")
    .reset_index()
    .rename(columns={"index": "columna"})
)

display(null_report)

# Comprobacion de la variable hora
hour_unique_count = df["snapshot_hour"].nunique(dropna=False)

if hour_unique_count == 1:
    print("REVISION: snapshot_hour solo tiene un valor. Habria que regenerar el dataset antes de cerrar el modelo.")
else:
    print("OK: snapshot_hour contiene varias horas y permite estudiar patrones horarios.")

,columna,pct_nulos
0,station_id_historical,0.0
1,station_latitude,0.0
2,station_longitude,0.0
3,station_capacity,0.0
4,num_bikes_available,0.0
5,snapshot_hour,0.0
6,snapshot_day_of_week,0.0
7,snapshot_month,0.0
8,day_type,0.0
9,is_working_day,0.0


OK: snapshot_hour contiene varias horas y permite estudiar patrones horarios.


## 4. Muestra de trabajo y division train/test

El dataset completo tiene mas de dos millones de filas. Para que esta comparacion sea rapida y reproducible, usamos una muestra aleatoria de 120.000 filas. Despues separamos en entrenamiento y prueba.

In [ ]:
# Muestra reproducible para acelerar el modelado inicial
if len(df) > SAMPLE_SIZE:
    df_model = df.sample(n=SAMPLE_SIZE, random_state=RANDOM_STATE).reset_index(drop=True)
else:
    df_model = df.copy().reset_index(drop=True)

# Separacion de variables predictoras y target
X = df_model[numeric_features + categorical_features]
y = df_model[TARGET]

# Train/test split para evaluar con datos no vistos
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
)

print(f"Filas usadas en la muestra: {len(df_model):,}")
print(f"Train: {len(X_train):,} filas")
print(f"Test: {len(X_test):,} filas")

Filas usadas en la muestra: 120,000
Train: 96,000 filas
Test: 24,000 filas


## 5. Preprocesamiento

Creamos dos preprocesadores:

- Uno para modelos lineales, con imputacion, escalado y One-Hot Encoding.
- Otro para Random Forest, sin escalado y con Ordinal Encoding para que el entrenamiento sea mas ligero.

In [ ]:
# Preprocesamiento para modelos lineales
linear_preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]
            ),
            numeric_features,
        ),
        (
            "cat",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("encoder", OneHotEncoder(handle_unknown="ignore")),
                ]
            ),
            categorical_features,
        ),
    ]
)

# Preprocesamiento para Random Forest
forest_preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline(steps=[("imputer", SimpleImputer(strategy="median"))]),
            numeric_features,
        ),
        (
            "cat",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
                ]
            ),
            categorical_features,
        ),
    ]
)

## 6. Definicion de modelos

Comparamos un baseline y tres modelos reales. El baseline es importante porque nos dice si los modelos estan aportando valor o si solo parecen buenos por casualidad.

In [ ]:
# Modelos que se van a entrenar y comparar
models = {
    "Baseline media": Pipeline(
        steps=[
            ("preprocesamiento", linear_preprocessor),
            ("modelo", DummyRegressor(strategy="mean")),
        ]
    ),
    "Regresion lineal": Pipeline(
        steps=[
            ("preprocesamiento", linear_preprocessor),
            ("modelo", LinearRegression()),
        ]
    ),
    "Ridge": Pipeline(
        steps=[
            ("preprocesamiento", linear_preprocessor),
            ("modelo", Ridge(alpha=1.0)),
        ]
    ),
    "Random Forest": Pipeline(
        steps=[
            ("preprocesamiento", forest_preprocessor),
            (
                "modelo",
                RandomForestRegressor(
                    n_estimators=40,
                    max_depth=18,
                    min_samples_leaf=5,
                    random_state=RANDOM_STATE,
                    n_jobs=-1,
                ),
            ),
        ]
    ),
}

## 7. Entrenamiento y evaluacion

Entrenamos cada modelo y calculamos:

- **MAE**: error medio absoluto en bicicletas.
- **RMSE**: error que penaliza mas los fallos grandes.
- **R2**: proporcion de variabilidad explicada por el modelo.

In [ ]:
# Funcion auxiliar para calcular metricas de regresion
def regression_metrics(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
    }

results = []
fitted_models = {}

# Entrenamiento y evaluacion de cada modelo
for model_name, model in models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    metrics = regression_metrics(y_test, predictions)

    results.append(
        {
            "modelo": model_name,
            "MAE": metrics["MAE"],
            "RMSE": metrics["RMSE"],
            "R2": metrics["R2"],
        }
    )
    fitted_models[model_name] = model

# Tabla ordenada por RMSE: cuanto menor, mejor
metrics_df = pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)
metrics_df

,modelo,MAE,RMSE,R2
0,Random Forest,3.632558,4.551091,0.426758
1,Ridge,4.394096,5.374670,0.200515
2,Regresion lineal,4.393797,5.374758,0.200489
3,Baseline media,4.987196,6.011013,-0.000005


## 8. Predicciones de ejemplo

Usamos el mejor modelo segun RMSE para ver ejemplos concretos. Esto permite explicar mejor que significan los errores en terminos de bicicletas.

In [ ]:
# Mejor modelo segun RMSE
best_model_name = metrics_df.loc[0, "modelo"]
best_model = fitted_models[best_model_name]

# Algunas filas de test para comparar valor real vs prediccion
example_rows = X_test.head(8).copy()
example_predictions = best_model.predict(example_rows)

prediction_examples = example_rows[[
    "station_id_historical",
    "snapshot_month",
    "snapshot_hour",
    "day_type",
]].copy()

prediction_examples["bicicletas_reales"] = y_test.head(8).to_numpy()
prediction_examples["bicicletas_predichas"] = np.round(example_predictions, 2)
prediction_examples["error"] = np.round(
    prediction_examples["bicicletas_predichas"] - prediction_examples["bicicletas_reales"],
    2,
)

print(f"Mejor modelo segun RMSE: {best_model_name}")
prediction_examples

Mejor modelo segun RMSE: Random Forest


,station_id_historical,snapshot_month,snapshot_hour,day_type,bicicletas_reales,bicicletas_predichas,error
71787,169,12,18,laborable,3,9.12,6.12
67218,101,6,13,sabado,6,11.51,5.51
54066,110,7,8,laborable,6,4.67,-1.33
7168,125,8,19,laborable,12,11.61,-0.39
29618,248,2,9,laborable,2,9.30,7.30
101425,187,3,10,domingo,8,15.23,7.23
20441,228,5,15,domingo,17,15.64,-1.36
2662,253,11,18,laborable,15,9.77,-5.23


## 9. Guardado del modelo final

Guardamos directamente el mejor modelo segun RMSE. En esta ejecucion, el mejor modelo es Random Forest, por lo que se exporta como artefacto de modelado para que pueda utilizarse en la app o en fases posteriores.

In [ ]:
# Guardado directo del mejor modelo
MODEL_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(best_model, MODEL_OUTPUT_PATH)

print(f"Mejor modelo guardado: {best_model_name}")
print(f"Ruta del modelo final: {MODEL_OUTPUT_PATH}")

Mejor modelo guardado: Random Forest
Ruta del modelo final: c:\Users\elena\OneDrive\Documentos\BOOTCAMP IA\proyecto1_modulo3\bike-sharing-prediction\models\modelo_final_definitivo_random_forest.joblib


## 10. Conclusiones

Con la muestra usada, **Random Forest** es el mejor modelo porque mejora claramente el baseline y obtiene el menor RMSE.

La correccion de `snapshot_hour` es importante: ahora el modelo puede aprender diferencias entre franjas horarias. En los ejemplos de prediccion ya aparecen horas distintas, lo que hace que el resultado sea mucho mas coherente con la pregunta de negocio de BiciMAD.

El modelo final definitivo queda guardado en `models/modelo_final_definitivo_random_forest.joblib`. Este artefacto corresponde a la parte de modelado y puede ser usado como base para la app o para una evaluacion posterior con mas modelos/hiperparametros.